# 稀疏矩阵：从短信文本到 CSR

本 Notebook 使用本地 UCI SMS Spam Collection 快照，手写确定性的 token、词表、TF-IDF 与 CSR 检查。目标不是训练垃圾短信分类器，而是观察真实文本怎样形成一个“行数不算夸张、列数很多、绝大多数位置为零”的矩阵。

固定约定：token 正则为 `[a-z0-9']+`，最低文档频率为 5，IDF 为 `log((1+n)/(1+df))+1`，最后逐行做 L2 归一化。

In [1]:
from collections import Counter
from pathlib import Path
import hashlib
import json
import os
import re

import numpy as np
import pandas as pd
from scipy import sparse

EXPECTED_SHA256 = "d43a9b9fe1530f4cc58a1e01ad23ee466283c9abce4a83be17b199899bd584f8"
TOKEN_PATTERN = re.compile(r"[a-z0-9']+")
MIN_DOCUMENT_FREQUENCY = 5

def locate_dataset():
    candidates = [
        Path("sms-spam.csv"),
        Path(os.environ["ML_ATLAS_SMS_DATA_PATH"]) if os.environ.get("ML_ATLAS_SMS_DATA_PATH") else None,
        Path("public/datasets/numerical-methods/sms-spam.csv"),
    ]
    for candidate in candidates:
        if candidate is not None and candidate.is_file():
            return candidate
    raise FileNotFoundError("Place sms-spam.csv beside this Notebook or set ML_ATLAS_SMS_DATA_PATH")

dataset_path = locate_dataset()
observed_sha = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
assert observed_sha == EXPECTED_SHA256
sms = pd.read_csv(dataset_path)
assert list(sms.columns) == ["sms_id", "label", "message"]
assert len(sms) == 5574
print(f"dataset={dataset_path.name} · rows={len(sms)} · sha256={observed_sha[:12]}…")

dataset=sms-spam.csv · rows=5574 · sha256=d43a9b9fe153…


## 1. 先定义“列是什么”

每条短信是一行，每个保留 token 是一列。词表必须先固定，矩阵的列坐标才有稳定含义。这里使用全语料建立表示，只讲存储结构，不做训练/测试评估。

In [2]:
documents = [TOKEN_PATTERN.findall(message.lower()) for message in sms["message"]]
document_frequency = Counter()
for tokens in documents:
    document_frequency.update(set(tokens))

vocabulary = sorted(
    token for token, frequency in document_frequency.items()
    if frequency >= MIN_DOCUMENT_FREQUENCY
)
token_to_column = {token: index for index, token in enumerate(vocabulary)}
print(f"documents = {len(documents)}")
print(f"vocabulary = {len(vocabulary)} tokens (minimum df = {MIN_DOCUMENT_FREQUENCY})")
print("first 12 columns =", vocabulary[:12])

documents = 5574
vocabulary = 1881 tokens (minimum df = 5)
first 12 columns = ["'", "''", "''ok''", "'ll", '00', '000', '02', '03', '04', '06', '0800', '08000839402']


## 2. 从非零计数构造 CSR

对每条短信只记录实际出现的词。CSR 的 `indptr[i]:indptr[i+1]` 给出第 `i` 行在 `indices` 和 `data` 中的窗口，不需要扫描整行词表。

In [3]:
rows, columns, values = [], [], []
for row_index, tokens in enumerate(documents):
    counts = Counter(token for token in tokens if token in token_to_column)
    for token in sorted(counts, key=token_to_column.__getitem__):
        rows.append(row_index)
        columns.append(token_to_column[token])
        values.append(float(counts[token]))

counts_csr = sparse.csr_matrix(
    (values, (rows, columns)),
    shape=(len(documents), len(vocabulary)),
    dtype=np.float64,
)
counts_csr.sort_indices()

df = np.asarray((counts_csr > 0).sum(axis=0)).ravel()
idf = np.log((1 + counts_csr.shape[0]) / (1 + df)) + 1
tfidf = counts_csr.multiply(idf).tocsr()
row_norms = np.sqrt(np.asarray(tfidf.multiply(tfidf).sum(axis=1)).ravel())
safe_norms = np.where(row_norms == 0, 1.0, row_norms)
tfidf = sparse.diags(1 / safe_norms) @ tfidf
tfidf = tfidf.tocsr()

print("shape =", list(tfidf.shape))
print("nnz =", tfidf.nnz)
print("indptr length =", len(tfidf.indptr))
print("all non-empty row norms ≈ 1:", np.max(np.abs(np.sqrt(np.asarray(tfidf.multiply(tfidf).sum(axis=1)).ravel())[row_norms > 0] - 1)))

shape = [5574, 1881]
nnz = 69798
indptr length = 5575
all non-empty row norms ≈ 1: 3.3306690738754696e-16


## 3. Dense 与 CSR 的真实存储差距

Dense `float64` 会给每个“短信 × token”位置分配 8 字节。CSR 只保存非零值、列索引和行指针。下面用 NumPy/SciPy 数组真实的 `nbytes` 计算，不把索引开销藏起来。

In [4]:
dense_bytes = tfidf.shape[0] * tfidf.shape[1] * np.dtype(np.float64).itemsize
csr_bytes = tfidf.data.nbytes + tfidf.indices.nbytes + tfidf.indptr.nbytes
density = tfidf.nnz / (tfidf.shape[0] * tfidf.shape[1])
compression = dense_bytes / csr_bytes

print(f"density = {density:.8f} ({density * 100:.5f}%)")
print(f"dense float64 = {dense_bytes / 1024**2:.3f} MiB")
print(f"CSR arrays = {csr_bytes / 1024**2:.3f} MiB")
print(f"dense / CSR = {compression:.2f}x")

density = 0.00665713 (0.66571%)
dense float64 = 79.992 MiB
CSR arrays = 0.820 MiB
dense / CSR = 97.55x


## 4. 手动扫描一行，与库运算对齐

下面对固定行手动遍历 `indptr` 窗口，计算和一个确定性向量的点积，再与 SciPy 的矩阵向量乘法结果比较。

In [5]:
probe = np.sin(np.arange(tfidf.shape[1], dtype=float) * 0.017)
row_index = 17
start, end = tfidf.indptr[row_index], tfidf.indptr[row_index + 1]
manual = 0.0
for entry_index in range(start, end):
    manual += tfidf.data[entry_index] * probe[tfidf.indices[entry_index]]
library = float((tfidf @ probe)[row_index])
manual_difference = abs(manual - library)

top_df_columns = np.argsort(-df, kind="stable")[:8]
top_document_frequency = [
    {"token": vocabulary[int(column)], "documents": int(df[int(column)])}
    for column in top_df_columns
]
print(f"row {row_index} window = [{start}, {end}) · entries = {end - start}")
print(f"manual = {manual:.12f} · SciPy = {library:.12f}")
print(f"absolute difference = {manual_difference:.3e}")
print("highest document-frequency tokens =", top_document_frequency)

row 17 window = [283, 299) · entries = 16
manual = -0.497805955961 · SciPy = -0.497805955961
absolute difference = 0.000e+00
highest document-frequency tokens = [{'token': 'to', 'documents': 1687}, {'token': 'i', 'documents': 1686}, {'token': 'you', 'documents': 1539}, {'token': 'a', 'documents': 1180}, {'token': 'the', 'documents': 1035}, {'token': 'u', 'documents': 831}, {'token': 'in', 'documents': 810}, {'token': 'and', 'documents': 795}]


In [6]:
summary = {
    "contractVersion": "numerical-methods-batch-2-v1",
    "outputId": "sms-sparse-summary",
    "datasetSha256": EXPECTED_SHA256,
    "rows": int(tfidf.shape[0]),
    "columns": int(tfidf.shape[1]),
    "nnz": int(tfidf.nnz),
    "density": float(density),
    "averageNonzerosPerRow": float(tfidf.nnz / tfidf.shape[0]),
    "denseBytesFloat64": int(dense_bytes),
    "csrBytes": int(csr_bytes),
    "denseToCsrRatio": float(compression),
    "manualRow": {
        "rowIndex": row_index,
        "start": int(start),
        "end": int(end),
        "entries": int(end - start),
        "manualDot": float(manual),
        "libraryDot": float(library),
        "absoluteDifference": float(manual_difference),
    },
    "topDocumentFrequency": top_document_frequency,
    "labelCounts": {key: int(value) for key, value in sms["label"].value_counts().sort_index().items()},
}

output_dir = Path(os.environ.get("ML_ATLAS_NUMERICAL_BATCH2_OUTPUT_DIR", "batch-2-outputs"))
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "sms-sparse-summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "contractVersion": "numerical-methods-batch-2-v1",
  "outputId": "sms-sparse-summary",
  "datasetSha256": "d43a9b9fe1530f4cc58a1e01ad23ee466283c9abce4a83be17b199899bd584f8",
  "rows": 5574,
  "columns": 1881,
  "nnz": 69798,
  "density": 0.006657132768967793,
  "averageNonzerosPerRow": 12.522066738428418,
  "denseBytesFloat64": 83877552,
  "csrBytes": 859876,
  "denseToCsrRatio": 97.54610199610177,
  "manualRow": {
    "rowIndex": 17,
    "start": 283,
    "end": 299,
    "entries": 16,
    "manualDot": -0.49780595596063804,
    "libraryDot": -0.49780595596063804,
    "absoluteDifference": 0.0
  },
  "topDocumentFrequency": [
    {
      "token": "to",
      "documents": 1687
    },
    {
      "token": "i",
      "documents": 1686
    },
    {
      "token": "you",
      "documents": 1539
    },
    {
      "token": "a",
      "documents": 1180
    },
    {
      "token": "the",
      "documents": 1035
    },
    {
      "token": "u",
      "documents": 831
    },
    {
      "tok